### Silver – Users

#### Purpose
Transform the Bronze `users` table into a clean and analytics-ready
Silver table by:
- Standardizing column names using a reusable UDF
- Enforcing correct data types
- Handling nulls based on business rules
- Deduplicating records
- Isolating malformed records into a quarantine table

#### Source
- coffee.bronze.users

#### Targets
- coffee.silver.users
- coffee.silver.quarantine_users


In [0]:
%python
# Widgets allow the same notebook to be executed across environments (DEV/PROD)
# and reused across multiple tables by changing only job parameters.
#
# default_watermark is used when the Silver table is empty (first run),
# enabling incremental ingestion logic without special casing.

dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("source_table", "users")
dbutils.widgets.text("default_watermark", "1900-01-01")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
source_table = dbutils.widgets.get("source_table")
default_watermark = dbutils.widgets.get("default_watermark")


In [0]:
%run ./Silver_utils/silver_transform_utils

In [0]:
%python
df_bronze = spark.table(f"{catalog}.{bronze_schema}.{source_table}")


In [0]:
%python
# Standardize all incoming column names using the central UDF.
# This ensures consistent snake_case naming across all Silver tables,
# regardless of how the raw files were named in Bronze.

df_std = standardize_columns(df_bronze)


In [0]:
%python
df_std.createOrReplaceTempView(f"bronze_{source_table}_std")


In [0]:
%python
# creating silver table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}  (
  user_id INT,
  gender STRING,
  birthdate DATE,
  registered_at TIMESTAMP,

  -- Bronze metadata
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP,
  load_dt DATE,
  source_file STRING,
  source_table STRING,

  -- Silver audit
  silver_loaded_at TIMESTAMP,
  silver_updated_at TIMESTAMP
)
USING DELTA
""")


In [0]:
-- Incremental extraction:
-- Only process Bronze rows that arrived after the latest loaded_at timestamp
-- already present in the Silver target table.
--
-- This prevents reprocessing old Bronze records and keeps Silver rerun-safe.

CREATE OR REPLACE TEMP VIEW bronze_users_incremental AS
SELECT *
FROM bronze_users_std
WHERE loaded_at >
(
  SELECT COALESCE(MAX(loaded_at), '1900-01-01')
  FROM coffee.silver.users
);


In [0]:
%python

#  Count invalid records for users

# For users, the minimum required column is user_id.
# registered_at is also important as a business timestamp.

invalid_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM bronze_users_incremental
WHERE
  user_id IS NULL
  OR registered_at IS NULL
""").collect()[0]["cnt"]

print("Invalid user rows:", invalid_count)


In [0]:
%python
# ---------------------------------------------
#  Create and load users quarantine table only if invalid rows exist
# ---------------------------------------------

if invalid_count > 0:

    
    #  Create users quarantine table
   
    spark.sql("""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{source_table}_quarantine (
      user_id STRING,
      gender STRING,
      birthdate STRING,
      registered_at STRING,

      -- Bronze metadata
      loaded_at TIMESTAMP,
      updated_at TIMESTAMP,
      load_dt DATE,
      source STRING,
      source_file STRING,

      -- Quarantine metadata
      quarantine_reason STRING,
      quarantined_at TIMESTAMP
    )
    USING DELTA
    """)

    
    # Merge invalid rows into quarantine (idempotent)
    
    spark.sql("""
    MERGE INTO {catalog}.{silver_schema}.{source_table}_quarantine q
    USING (
      SELECT
        *,
        CASE
          WHEN user_id IS NULL THEN 'user_id is null'
          WHEN registered_at IS NULL THEN 'registered_at is null'
          ELSE 'unknown validation failure'
        END AS quarantine_reason,
        current_timestamp() AS quarantined_at
      FROM bronze_users_incremental
      WHERE
        user_id IS NULL
        OR registered_at IS NULL
    ) b
    ON q.user_id = b.user_id
    AND q.quarantine_reason = b.quarantine_reason
    WHEN NOT MATCHED THEN
    INSERT *;
    """)

else:
    print("No invalid user rows found. Quarantine table not created.")


In [0]:
%python
spark.sql(f""" 
MERGE INTO  {catalog}.{silver_schema}.{source_table} s
USING (

  SELECT
    TRY_CAST(user_id AS INT)             AS user_id,
    LOWER(gender)                        AS gender,
    TRY_CAST(birthdate AS DATE)          AS birthdate,
    TRY_CAST(registered_at AS TIMESTAMP) AS registered_at,

    loaded_at,
    updated_at,
    load_dt,
    source_file,
    'coffee.bronze.users' AS source_table,

    current_timestamp() AS silver_updated_at
-- Deduplication logic:
-- Bronze may contain duplicates for the same business key.
-- We keep only the latest version of each record using:
--   ROW_NUMBER() OVER (PARTITION BY <business_key> ORDER BY updated_at DESC)
--
-- This ensures Silver contains a single clean record per business key.

  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY user_id
             ORDER BY updated_at DESC
           ) AS rn
    FROM bronze_users_incremental
    WHERE
      user_id IS NOT NULL
      AND registered_at IS NOT NULL
  )
  WHERE rn = 1

) b

ON s.user_id = b.user_id

WHEN MATCHED THEN
  UPDATE SET
    s.gender            = b.gender,
    s.birthdate         = b.birthdate,
    s.registered_at     = b.registered_at,
    s.updated_at        = b.updated_at,
    s.load_dt           = b.load_dt,
    s.source_file       = b.source_file,
    s.silver_updated_at = b.silver_updated_at

WHEN NOT MATCHED THEN
  INSERT (
    user_id,
    gender,
    birthdate,
    registered_at,
    loaded_at,
    updated_at,
    load_dt,
    source_file,
    source_table,
    silver_loaded_at,
    silver_updated_at
  )
  VALUES (
    b.user_id,
    b.gender,
    b.birthdate,
    b.registered_at,
    b.loaded_at,
    b.updated_at,
    b.load_dt,
    b.source_file,
    b.source_table,
    current_timestamp(),
    current_timestamp()
  )
  """)
